# Analyses of the RNNs trained in PRE and POST

Networks trained with `training_PRE_POST.ipynb`.
Run in the `NEURCOMP` virtual environment (`../environment.yml`).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
import pickle
import platform
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import ttest_rel, t

sys.path.insert(0, os.path.abspath('..'))  # func/ is in python/
from func import *

%matplotlib inline
plt.style.use('default')
plt.rc('axes', facecolor='w')
plt.rc('figure', facecolor='w')

print(platform.platform())
print(sys.executable)  # should be the NEURCOMP environment
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

condtn = ['_PRE', '_POST', '_REF']  # PRE:0; POST:1; REF:2
rgb_PRE  = (0.0000, 0.4470, 0.7410)
rgb_POST = (0.8500, 0.3250, 0.0980)
rgb_REF  = (.7, .7, .7)

In [ ]:
# load the trained networks (one .tar per network) and the networks saved every savecheck epochs (.pt)
sigcat = 15
noise_val = 0
noise_str = f'{noise_val:.2f}'.replace('.', '-')
fname_train = os.path.join('..', 'rnn_simulations', 'RNN_train_PRE_POST', f'sigcat_{sigcat}' + (f'_noise_{noise_str}' if noise_val != 0 else ''))

file_list = sorted(glob.glob(os.path.join(fname_train, 'model_irnn_*.tar')))  # sorted by name = by network number
irnn_list = [int(os.path.basename(file).split('_')[2][:-4]) for file in file_list]  # network numbers (seeds)
file_list_epo = []
acc_train_PRE, acc_train_POST = [], []  # accuracy along training (%, last batch of each epoch)
acc_train_opt = []                      # accuracy of the optimal observer on the same batches (%)
for file in file_list:
    file_list_epo.append(sorted(glob.glob(file[:-4] + '_epo_*.pt')))  # 4-digit epochs: sorted by name = by epoch
    net = torch.load(file, weights_only=False)  # saved network: cfg and training history
    acc_train_PRE.append(net['history_train']['PRE'])
    acc_train_POST.append(net['history_train']['POST'])
    acc_train_opt.append(net['history_train']['opt'])
acc_train_PRE, acc_train_POST, acc_train_opt = np.array(acc_train_PRE), np.array(acc_train_POST), np.array(acc_train_opt)
cfg = net['cfg']  # same training configuration for all networks

N = len(file_list)
nepo = len(file_list_epo[0])  # number of saved epochs
epochs = np.array([int(f[-7:-3]) for f in file_list_epo[0]])  # saved epochs (0: untrained network)
print(f'{N} networks (sigcat = {sigcat}, noise = {noise_val}), {nepo} saved epochs each ({epochs[0]} to {epochs[-1]} epochs)')
if any(len(f) != nepo for f in file_list_epo):
    print('Warning: not the same number of saved epochs for all networks!')

## 1. Along training (Figure 7b)

Every saved epoch of each network is tested on the same equalized sequences:
`nseq` sequences of 4, 8 and 12 stimuli, each one tested in PRE and in POST.
The variables are decoded from the hidden activity of all stimuli of all sequences pooled
(`decode_ridg`, as for the MEG data), separately in each condition.

In [ ]:
# configuration of the tests (all sections): each network is always tested on its own sequences
seq2test   = [4, 8, 12]   # numbers of stimuli
nseq       = 1000         # sequences per number of stimuli (each one tested in PRE and in POST)
seed_test  = 1            # sequences of network irnn: SeedSequence(seed_test, spawn_key=(irnn,)), not the training seeds
condn_test = [0, 1]       # PRE, POST
var2decode = ['cumllr', 'cumnul', 'cumsumcos', 'cumsumsin']
seeds = [int(np.random.SeedSequence(seed_test, spawn_key=(irnn,)).generate_state(1)[0]) for irnn in irnn_list]
seeds_new = [int(np.random.SeedSequence(seed_test, spawn_key=(irnn, 1)).generate_state(1)[0]) for irnn in irnn_list]  # other sequences of network irnn

In [ ]:
# test every saved epoch of each network on the same sequences in both conditions, decode, and save
fname_test = os.path.join(fname_train, 'test_epochs.pkl')
out_epo = Parallel(n_jobs=-1)(
    delayed(run_simulations)(file_list_epo[i], cfg, seq2test, nseq, seeds[i], device, var2decode, condn_test)
    for i in range(N))
with open(fname_test, 'wb') as f:
    pickle.dump({'out_epo': out_epo, 'irnn_list': irnn_list, 'epochs': epochs, 'seq2test': seq2test,
                 'nseq': nseq, 'seed_test': seed_test, 'condn_test': condn_test, 'var2decode': var2decode}, f)
print(f'saved in {fname_test}')

In [ ]:
# load the saved results and gather them: (N, nepo, 2) for PRE and POST
fname_test = os.path.join(fname_train, 'test_epochs.pkl')
with open(fname_test, 'rb') as f:
    res_test = pickle.load(f)
out_epo    = res_test['out_epo']
var2decode = res_test['var2decode']
epochs     = res_test['epochs']  # tested epochs
nepo       = len(epochs)

acc       = np.array([o['pcor'] for o in out_epo])        # decision accuracy
acc_bayes = np.array([o['pcor_bayes'] for o in out_epo])  # accuracy of the Bayes-optimal observer on the same sequences
rdec      = {v: np.array([o['rdec'][v] for o in out_epo]) for v in var2decode}  # decoding r
compression = rdec['cumllr'] - rdec['cumnul']
cumang      = (rdec['cumsumcos'] + rdec['cumsumsin']) / 2  # coding of the mean orientation
print(f'Bayes-optimal accuracy: {np.mean(acc_bayes):.3f}')
acc_end = acc[:, -1, :]  # (N, 2)
acc_end_sem = np.std(acc_end, axis=0, ddof=1) / np.sqrt(N)
print(f'epoch {epochs[-1]}: accuracy PRE = {np.mean(acc_end[:, 0]):.4f} ± {acc_end_sem[0]:.4f}, '
      f'POST = {np.mean(acc_end[:, 1]):.4f} ± {acc_end_sem[1]:.4f} (mean ± s.e.m., N = {N})')
print(f'epoch {epochs[-1]}: compression PRE = {np.mean(compression[:, -1, 0]):.3f}, POST = {np.mean(compression[:, -1, 1]):.3f}')

In [ ]:
# accuracy along training: PRE vs POST, cluster-based permutation test on the paired difference
stat = cluster_permutation(acc[:, :, 0] - acc[:, :, 1], None, alpha_form=0.05, nperm=10000, seed=1234, xvec=epochs)
for c in stat['clusters']:
    dz = np.mean(stat['t'][c['idx']]) / np.sqrt(N)  # Cohen's dz of the cluster
    print(f"epochs {c['xbeg']}-{c['xend']}: {'PRE > POST' if c['tstat'] > 0 else 'POST > PRE'}, dz = {dz:+.2f}, p = {c['p']:.4f}")

# single networks (light), mean (thick), chance (grey), Bayes-optimal accuracy (dotted)
fig, ax = plt.subplots(figsize=(4.5, 4))
for icond, rgb in enumerate((rgb_PRE, rgb_POST)):
    ax.plot(epochs, acc[:, :, icond].T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
ax.axhline(0.5, color=(.8, .8, .8))
for icond, rgb in enumerate((rgb_PRE, rgb_POST)):
    ax.plot(epochs, np.mean(acc[:, :, icond], axis=0), color=rgb, linewidth=2)
ax.axhline(np.mean(acc_bayes), color='k', linestyle=':')
# significant clusters (p < 0.05) on top of the plot: PRE > POST in blue, POST > PRE in orange
for c in stat['clusters']:
    if c['p'] < 0.05:
        ax.plot([c['xbeg'], c['xend']], [0.97, 0.97], color=rgb_PRE if c['tstat'] > 0 else rgb_POST, linewidth=3)
ax.set_ylim(0.4, 1)
ax.set_xticks(np.arange(200, epochs[-1] + 1, 200))
ax.set_xlabel('training epoch')
ax.set_ylabel('decision accuracy')
plt.tight_layout()

In [ ]:
# compression (r cumllr - r cumnul) along training: single networks (light), mean (thick)
fig, ax = plt.subplots(figsize=(4.5, 4))
for icond, rgb in enumerate((rgb_PRE, rgb_POST)):
    ax.plot(epochs, compression[:, :, icond].T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
ax.axhline(0, color=(.8, .8, .8))
for icond, rgb in enumerate((rgb_PRE, rgb_POST)):
    ax.plot(epochs, np.mean(compression[:, :, icond], axis=0), color=rgb, linewidth=2)
ax.set_xticks(np.arange(200, epochs[-1] + 1, 200))
ax.set_xlabel('training epoch')
ax.set_ylabel('compression')
plt.tight_layout()

In [ ]:
# partial correlation across networks between compression and accuracy, controlling for the coding
# of the mean orientation (cumang), in PRE; cluster-based permutation test (permutation of networks)
icond = 0
X, Y, Z = compression[:, :, icond], acc[:, :, icond], cumang[:, :, icond]  # (N, nepo)
stat_par = cluster_permutation(Y, X, alpha_form=0.05, nperm=10000, seed=1234, xvec=epochs, Z=Z)
r_par = stat_par['r']
for c in stat_par['clusters']:
    print(f"epochs {c['xbeg']}-{c['xend']}: mean partial r = {np.mean(r_par[c['idx']]):.2f}, p = {c['p']:.4f}")

# s.e.m. of the partial correlation: bootstrap over networks
nboot = 10000
rng = np.random.default_rng(1234)
r_boot = np.zeros((nboot, nepo))
for iboot in range(nboot):
    jboot = rng.integers(0, N, N)
    r_boot[iboot] = partial_corr(X[jboot], Y[jboot], Z[jboot])
r_sem = np.std(r_boot, axis=0, ddof=1)

fig, ax = plt.subplots(figsize=(4.5, 2.5))
ax.fill_between(epochs, r_par - r_sem, r_par + r_sem, color=(1 + np.array(rgb_PRE)) / 2, alpha=0.5, linewidth=0)
ax.axhline(0, color=(.8, .8, .8))
# significant clusters (p < 0.05) on the zero line
for c in stat_par['clusters']:
    if c['p'] < 0.05:
        ax.plot([c['xbeg'], c['xend']], [0, 0], color=rgb_PRE, linewidth=3)
ax.plot(epochs, r_par, color=rgb_PRE, linewidth=2)
ax.set_xticks(np.arange(200, epochs[-1] + 1, 200))
ax.set_xlabel('training epoch')
ax.set_ylabel('partial correlation')
plt.tight_layout()

In [ ]:
# export to .mat (plots in MATLAB): (N networks, epochs, conditions PRE and POST); indices of the clusters are 1-based
export_mat(os.path.join(fname_train, 'test_epochs.mat'), sigcat=sigcat, irnn=np.array(irnn_list),
           epochs=epochs, acc=acc, acc_bayes=acc_bayes, rdec=rdec, compression=compression, cumang=cumang,
           stat_acc=stat, r_par=r_par, r_sem=r_sem, stat_par=stat_par)

## 2. Resilience to computation noise (Figure 7c)

The trained networks (last epoch) are tested on the same sequences as in section 1,
with isotropic noise added to the pre-activation of the hidden units after each stimulus
(not at the final step: POST-cue and GO signal), for each noise level.
With the same seed, the noise samples are the same for all noise levels, only scaled by the noise level.

In [ ]:
# test the trained networks (last epoch) for each noise level, decode, and save
noise_array = np.round(np.arange(0, 1 + 1e-9, 0.02), 2)  # s.d. of the noise
fname_noise = os.path.join(fname_train, 'test_noise.pkl')
out_noise = Parallel(n_jobs=-1)(
    delayed(run_simulations)(file_list_epo[i][-1], cfg, seq2test, nseq, seeds[i], device, var2decode, condn_test,
                        noise_values=noise_array)
    for i in range(N))
with open(fname_noise, 'wb') as f:
    pickle.dump({'out_noise': out_noise, 'irnn_list': irnn_list, 'file_epo': [f[-1] for f in file_list_epo],
                 'noise_array': noise_array, 'seq2test': seq2test, 'nseq': nseq, 'seed_test': seed_test,
                 'condn_test': condn_test, 'var2decode': var2decode}, f)
print(f'saved in {fname_noise}')

In [ ]:
# load the saved results and gather them: (N, nnoise, 2) for PRE and POST
fname_noise = os.path.join(fname_train, 'test_noise.pkl')
with open(fname_noise, 'rb') as f:
    res_noise = pickle.load(f)
out_noise   = res_noise['out_noise']
var2decode  = res_noise['var2decode']
noise_array = res_noise['noise_array']

acc_noise = np.array([o['pcor'] for o in out_noise])  # decision accuracy
# decoding r on all sequences, on correct sequences and on error sequences
rdec_noise, rdec_noise_cor, rdec_noise_err = (
    {v: np.array([o[key][v] for o in out_noise]) for v in var2decode} for key in ('rdec', 'rdec_cor', 'rdec_err'))
for r in (rdec_noise, rdec_noise_cor, rdec_noise_err):
    r['compression'] = r['cumllr'] - r['cumnul']
    r['cumang'] = (r['cumsumcos'] + r['cumsumsin']) / 2  # coding of the mean orientation
print(f'{len(noise_array)} noise levels from {noise_array[0]} to {noise_array[-1]}')

In [ ]:
# accuracy difference PRE - POST as a function of noise: cluster-based permutation test on the paired difference,
# one-tailed (PRE > POST)
acc_diff = (acc_noise[:, :, 0] - acc_noise[:, :, 1]) * 100  # (%)
stat_noise = cluster_permutation(acc_diff, None, alpha_form=0.05, tail='right', nperm=10000, seed=1234, xvec=noise_array)
for c in stat_noise['clusters']:
    dz = np.mean(stat_noise['t'][c['idx']]) / np.sqrt(N)  # Cohen's dz of the cluster
    print(f"noise {c['xbeg']}-{c['xend']}: PRE > POST, dz = {dz:+.2f}, p = {c['p']:.4f}")

# mean +/- s.e.m. across networks
yavg = np.mean(acc_diff, axis=0)
yerr = np.std(acc_diff, axis=0, ddof=1) / np.sqrt(N)
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.fill_between(noise_array, yavg - yerr, yavg + yerr, color=(.8, .8, .8), linewidth=0)
ax.plot(noise_array, yavg, color=(.25, .25, .25), linewidth=1.5)
ax.axhline(0, color='k', linestyle=':')
# significant clusters (p < 0.05) on the zero line
for c in stat_noise['clusters']:
    if c['p'] < 0.05:
        ax.plot([c['xbeg'], c['xend']], [0, 0], color=rgb_PRE, linewidth=3)
ax.set_xlim(noise_array[0], noise_array[-1])
ax.set_xlabel('neural noise')
ax.set_ylabel('accuracy difference (%)')
plt.tight_layout()

In [ ]:
# mean accuracy (PRE and POST) as a function of noise: single networks (light), mean +/- s.e.m. (black)
acc_mean = np.mean(acc_noise, axis=2)  # (N, nnoise)
yavg = np.mean(acc_mean, axis=0)
yerr = np.std(acc_mean, axis=0, ddof=1) / np.sqrt(N)
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.plot(noise_array, acc_mean.T, color=(.8, .8, .8), linewidth=0.5)
ax.fill_between(noise_array, yavg - yerr, yavg + yerr, color=(.6, .6, .6), linewidth=0)
ax.plot(noise_array, yavg, color='k', linewidth=1.5)
ax.set_xlim(noise_array[0], noise_array[-1])
ax.set_ylim(0.5, 0.85)
ax.set_xlabel('neural noise')
ax.set_ylabel('mean accuracy')
plt.tight_layout()

In [ ]:
# coding precision on the correct (solid) and error (dashed) sequences as a function of noise
icond = 0  # PRE
rgb = (rgb_PRE, rgb_POST)[icond]
fig, axs = plt.subplots(3, 1, figsize=(4.5, 8), sharex=True)
for ax, (v, name) in zip(axs, (('cumang', 'mean orientation'), ('cumnul', 'null belief'), ('cumllr', 'true belief'))):
    y_cor = rdec_noise_cor[v][:, :, icond]
    y_err = rdec_noise_err[v][:, :, icond]
    ax.plot(noise_array, y_cor.T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
    ax.plot(noise_array, y_err.T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
    ax.axhline(0, color=(.8, .8, .8))
    ax.plot(noise_array, np.mean(y_cor, axis=0), color=rgb, linewidth=2, label='correct')
    ax.plot(noise_array, np.mean(y_err, axis=0), color=rgb, linewidth=2, linestyle='--', label='error')
    ax.set_ylim(-0.1, 1)
    ax.set_title(name)
    ax.set_ylabel(f'coding precision (r)')
    # difference between correct and error sequences, averaged over noise levels: mean +/- s.e.m. across networks
    d = np.mean(y_cor - y_err, axis=1)
    print(f'{name}: % difference between correct and error trials: {100 * np.mean(d):.3f} ± {100 * np.std(d, ddof=1) / np.sqrt(N):.3f} %')
axs[0].legend(frameon=False)
axs[-1].set_xlabel('neural noise')
axs[-1].set_xlim(noise_array[0], noise_array[-1])
plt.tight_layout()

In [ ]:
# export to .mat (plots in MATLAB): (N networks, noise levels, conditions PRE and POST); indices of the clusters are 1-based
export_mat(os.path.join(fname_train, 'test_noise.mat'), sigcat=sigcat, irnn=np.array(irnn_list),
           noise_array=noise_array, acc=acc_noise, rdec=rdec_noise, rdec_cor=rdec_noise_cor, rdec_err=rdec_noise_err,
           stat_diff=stat_noise)

## 3. Causal perturbation (Figure 7d)

For each trained network (last epoch) and each condition:
1. on the sequences of sections 1 and 2 (no noise), the decoding direction of each variable
   (`cumllr`, `cumnul`, `cumsumcos`, `cumsumsin`) is given by its decoder weights (`decode_ridg`, all sequences),
   normalized;
2. on new sequences, noise is injected along one direction at a time, in the pre-activation after each stimulus,
   with s.d. = noise level x s.d. of the pre-activation along this direction (one sample per sequence and stimulus,
   the same samples for all directions and levels);
3. accuracy and decoding (decoders retrained on the perturbed activity) are compared with the unperturbed network
   on the same new sequences, and averaged over the noise levels.

The mean orientation is perturbed along `cumsumcos` and along `cumsumsin` separately, and the two results are averaged.

In [ ]:
# causal perturbation of the trained networks (last epoch), decode, and save
sel_noise_array = np.round(np.arange(0, 5 + 1e-9, 0.5), 1)  # noise level (in s.d. of the pre-activation along the direction)
var2perturb = ['cumllr', 'cumnul', 'cumsumcos', 'cumsumsin']
fname_pert = os.path.join(fname_train, 'test_perturbation.pkl')
out_pert = Parallel(n_jobs=-1)(
    delayed(run_perturbation)(file_list_epo[i][-1], cfg, sel_noise_array, seq2test, nseq, seeds[i], seeds_new[i],
                              device, var2decode, var2perturb, condn_test)
    for i in range(N))
with open(fname_pert, 'wb') as f:
    pickle.dump({'out_pert': out_pert, 'irnn_list': irnn_list, 'file_epo': [f[-1] for f in file_list_epo],
                 'sel_noise_array': sel_noise_array, 'var2perturb': var2perturb, 'seq2test': seq2test, 'nseq': nseq,
                 'seed_test': seed_test, 'condn_test': condn_test, 'var2decode': var2decode}, f)
print(f'saved in {fname_pert}')

In [ ]:
# load the saved results
fname_pert = os.path.join(fname_train, 'test_perturbation.pkl')
with open(fname_pert, 'rb') as f:
    res_pert = pickle.load(f)
out_pert        = res_pert['out_pert']
var2decode      = res_pert['var2decode']
var2perturb     = res_pert['var2perturb']
sel_noise_array = res_pert['sel_noise_array']

# unperturbed networks: (N, 2) for PRE and POST
acc0  = np.array([o['pcor0'] for o in out_pert])
rdec0 = {v: np.array([o['rdec0'][v] for o in out_pert]) for v in var2decode}
# perturbation along the direction of each variable: (N, nlev, 2)
acc_pert  = {p: np.array([o['pcor'][p] for o in out_pert]) for p in var2perturb}
rdec_pert = {p: {v: np.array([o['rdec'][p][v] for o in out_pert]) for v in var2decode} for p in var2perturb}
for r in [rdec0] + list(rdec_pert.values()):
    r['compression'] = r['cumllr'] - r['cumnul']
    r['cumang'] = (r['cumsumcos'] + r['cumsumsin']) / 2  # coding of the mean orientation
# perturbation of the mean orientation: average of the perturbations along cumsumcos and along cumsumsin
acc_pert['cumang']  = (acc_pert['cumsumcos'] + acc_pert['cumsumsin']) / 2
rdec_pert['cumang'] = {v: (rdec_pert['cumsumcos'][v] + rdec_pert['cumsumsin'][v]) / 2 for v in rdec_pert['cumsumcos']}
# coding of the mean orientation under its own perturbation: True = coding of the perturbed component only
# (cumsumcos under the cumsumcos perturbation, cumsumsin under the cumsumsin perturbation),
# False = coding of both components (cumang), as for the other perturbations
cumang_matched = True
if cumang_matched:
    rdec_pert['cumang']['cumang'] = (rdec_pert['cumsumcos']['cumsumcos'] + rdec_pert['cumsumsin']['cumsumsin']) / 2

# effect of each perturbation, averaged over the noise levels, for each network: (N, 4)
bars     = [('cumang', 1), ('cumang', 0), ('cumllr', 0), ('cumnul', 0)]  # (perturbed variable, condition)
barnames = ['POST\nmean orient.', 'PRE\nmean orient.', 'PRE\ntrue belief', 'PRE\nnull belief']
barrgb   = [rgb_POST, rgb_PRE, rgb_PRE, rgb_PRE]
eff = {'accuracy drop (%)':
           np.column_stack([(np.mean(acc_pert[p][:, :, c], axis=1) - acc0[:, c]) / acc0[:, c] * 100 for p, c in bars]),
       'mean orient. coding change':
           np.column_stack([np.mean(rdec_pert[p]['cumang'][:, :, c], axis=1) - rdec0['cumang'][:, c] for p, c in bars]),
       'compression change':
           np.column_stack([np.mean(rdec_pert[p]['compression'][:, :, c], axis=1) - rdec0['compression'][:, c] for p, c in bars])}
for name, x in eff.items():
    print(name + ': ' + ' | '.join(f"{b.replace(chr(10), ' ')} {np.mean(x[:, i]):+.3f}" for i, b in enumerate(barnames)))

In [ ]:
# accuracy drop: POST mean orientation vs PRE mean orientation and vs PRE true belief (paired t-tests across networks)
x = eff['accuracy drop (%)']
for i in (1, 2):
    tval, pval = ttest_rel(x[:, 0], x[:, i])
    dz = np.mean(x[:, 0] - x[:, i]) / np.std(x[:, 0] - x[:, i], ddof=1)
    bf10 = bf_ttest(tval, N)
    if pval < 0.05:
        BFval, BFstr = bf10, 'BF10'
    else:
        BFval, BFstr = 1 / bf10, 'BF01'
    print(f"accuracy drop {barnames[0].replace(chr(10), ' ')} vs {barnames[i].replace(chr(10), ' ')}: "
          f"t({N - 1}) = {tval:+.3f}, p = {pval:.5f}, dz = {dz:+.2f}, {BFstr} = {BFval:.3f}")

In [ ]:
# bars: mean across networks, dots: single networks
fig, axs = plt.subplots(3, 1, figsize=(4.5, 8), sharex=True)
rng = np.random.default_rng(0)  # jitter of the dots
for ax, (name, x) in zip(axs, eff.items()):
    for i, rgb in enumerate(barrgb):
        ax.bar(i, np.mean(x[:, i]), width=0.6, color=(1 + np.array(rgb)) / 2, edgecolor=rgb)
        ax.plot(i + rng.uniform(-0.15, 0.15, N), x[:, i], 'o', color=rgb, markerfacecolor='w', markersize=4)
    ax.axhline(0, color='k', linestyle='--', linewidth=1)
    ax.set_ylabel(name)
axs[-1].set_xticks(range(len(barnames)))
axs[-1].set_xticklabels(barnames)
axs[-1].set_xlabel('perturbation')
plt.tight_layout()

In [ ]:
# export to .mat (plots in MATLAB): unperturbed (N networks, conditions PRE and POST),
# perturbed (N networks, noise levels, conditions), effects averaged over the noise levels (N networks, bars)
export_mat(os.path.join(fname_train, 'test_perturbation.mat'), sigcat=sigcat, irnn=np.array(irnn_list),
           cumang_matched=cumang_matched, sel_noise_array=sel_noise_array, acc0=acc0, rdec0=rdec0,
           acc_pert=acc_pert, rdec_pert=rdec_pert, barnames=[b.replace('\n', ' ') for b in barnames], eff=eff)